# Observing parameter access

QCoDeS can tell you about every `get` and `set` of every parameter in your process. You register an *observer*: any callable that takes a `ParameterAccess`. QCoDeS calls it after each access, with the parameter, the value, the raw value, when the access started, how long it took and the exception it raised, if any.

This lets you see what a piece of code did to your instruments without changing the code or the drivers it uses. You can record what a measurement set and read, find out which parameters are queried most often, or keep an audit log of everything that was set.

Observers are independent of a parameter's `on_set_callback` and of `ParameterBase.global_on_set_callback`. Any number of them can be registered, and they also see `get`.

In [1]:
import logging
import sys
import tempfile
from collections import Counter
from pathlib import Path

from qcodes.dataset import (
    do1d,
    initialise_or_create_database_at,
    load_or_create_experiment,
)
from qcodes.instrument_drivers.mock_instruments import (
    DummyInstrument,
    DummyInstrumentWithMeasurement,
)
from qcodes.parameters import (
    Parameter,
    ParameterAccess,
    add_parameter_observer,
    observe_parameters,
    registered_parameter_observers,
    remove_parameter_observer,
)

We use two mock instruments: a DAC with two channels, and a DMM whose readings depend on the DAC's voltages.

In [2]:
dac = DummyInstrument("dac", gates=["ch1", "ch2"])
dmm = DummyInstrumentWithMeasurement("dmm", setter_instr=dac)

## Recording the accesses of a block of code

`observe_parameters` registers an observer for the duration of a `with` block. Here the observer is the `append` method of a list, so the list collects every access made inside the block.

In [3]:
accesses: list[ParameterAccess] = []

with observe_parameters(accesses.append):
    dac.ch1(0.5)
    dmm.v1()

for access in accesses:
    print(f"{access.kind} {access.parameter.full_name}: {access.value}")

set dac_ch1: 0.5
get dmm_v1: 4.66492785682728


Accesses outside the block are not recorded:

In [4]:
dac.ch1(0.7)
len(accesses)

2

## What an access contains

Each `ParameterAccess` describes one access:

- `parameter`: the parameter that was read or written.
- `kind`: `"get"` or `"set"`.
- `value`: the value that was read or written.
- `raw_value`: the value passed to or returned by the instrument, before scaling, offsets or value mappings are applied.
- `timestamp`: when the access started, in seconds since the epoch.
- `duration`: how long the access took, in seconds. Delays configured on the parameter, such as `post_delay`, are not included.
- `exception`: the exception the access raised, or `None`.

The difference between `value` and `raw_value` shows what actually reached the instrument. For example, behind a 1:10 voltage divider, setting 2 V on the sample means setting 20 V on the source:

In [5]:
divided = Parameter("divided", unit="V", set_cmd=None, get_cmd=None, scale=10)

set_accesses: list[ParameterAccess] = []
with observe_parameters(set_accesses.append):
    divided(2.0)

(access,) = set_accesses
print(
    f"value={access.value}, raw_value={access.raw_value}, exception={access.exception}"
)

value=2.0, raw_value=20.0, exception=None


## Ramped sets are reported step by step

A parameter with a `step` is set in several steps, each of which is a separate call to the instrument. The observer is notified of every step:

In [6]:
dac.ch2.step = 0.1
dac.ch2.inter_delay = 0

ramp: list[ParameterAccess] = []
with observe_parameters(ramp.append):
    dac.ch2(0.5)

[round(access.value, 2) for access in ramp]

[0.1, 0.2, 0.3, 0.4, 0.5]

## Failed accesses

An access that raises is reported too, with the exception in `exception`, and the exception then propagates to the caller as usual. A value that fails validation is rejected before the instrument is contacted, so it is not an access and is not reported.

In [7]:
def read_unresponsive_sensor() -> float:
    raise TimeoutError("the sensor did not answer")


sensor = Parameter("sensor", get_cmd=read_unresponsive_sensor)

failures: list[ParameterAccess] = []
with observe_parameters(failures.append):
    try:
        sensor()
    except TimeoutError as error:
        print(f"The caller still sees the error: {error.args[0]}")

(failure,) = failures
print(
    f"The observer saw: {failure.kind} {failure.parameter.full_name}: {failure.exception!r}"
)

The caller still sees the error: the sensor did not answer
The observer saw: get sensor: TimeoutError('the sensor did not answer', 'getting sensor')


## What did a measurement do to the instruments?

Observers see all accesses, including those made by QCoDeS itself, so they can show what a measurement did. Here we run a 1D sweep and count the accesses per parameter.

In [8]:
initialise_or_create_database_at(Path(tempfile.mkdtemp()) / "observers.db")
load_or_create_experiment("observing_parameters", sample_name="no sample")

sweep: list[ParameterAccess] = []
with observe_parameters(sweep.append):
    do1d(dac.ch1, 0, 1, 11, 0.0, dmm.v1, do_plot=False)

Counter((access.parameter.full_name, access.kind) for access in sweep)

Starting experimental run with id: 1. Using 'qcodes.dataset.do1d'


Counter({('dac_ch1', 'set'): 11, ('dmm_v1', 'get'): 11})

The sweep set the DAC 11 times and read the DMM 11 times. The accesses are in the order they happened, so we can also follow the sweep point by point:

In [9]:
for access in sweep[:6]:
    print(f"{access.kind} {access.parameter.full_name}: {access.value:.3f}")

set dac_ch1: 0.000
get dmm_v1: 5.033
set dac_ch1: 0.100
get dmm_v1: 4.939
set dac_ch1: 0.200
get dmm_v1: 4.915


Observers see every parameter in the process. To follow one instrument, filter on the root instrument of the accessed parameter:

In [10]:
dac_sets = [
    access.value
    for access in sweep
    if access.parameter.root_instrument is dac and access.kind == "set"
]
[round(float(value), 2) for value in dac_sets]

[0.0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0]

## Observers that stay registered

`add_parameter_observer` registers an observer until it is removed with `remove_parameter_observer`. `registered_parameter_observers` returns the observers registered at the moment.

Here an observer writes every access to a logger:

In [11]:
access_log = logging.getLogger("parameter_access")
access_log.setLevel(logging.INFO)
access_log.propagate = False
access_log.addHandler(logging.StreamHandler(sys.stdout))


def log_access(access: ParameterAccess) -> None:
    access_log.info("%s %s = %r", access.kind, access.parameter.full_name, access.value)


add_parameter_observer(log_access)
log_access in registered_parameter_observers()

True

In [12]:
dac.ch1(0.25)
reading = dmm.v2()

set dac_ch1 = 0.25


get dmm_v2 = 0.4601170962709825


In [13]:
remove_parameter_observer(log_access)
dac.ch1(0)  # no longer logged
log_access in registered_parameter_observers()

False

## Good to know

- Observers are called synchronously, on the thread that accessed the parameter, right after the access. Keep them fast: an observer that takes a millisecond slows down every parameter access in the process by a millisecond. To do slow work, such as writing to a database, hand the access over to a queue.
- An exception raised by an observer is logged as a warning and ignored. It neither affects the access nor stops other observers from being notified.
- An access to a `DelegateParameter` is reported twice: once for the delegate and once for its source.
- Observers only see parameters. Commands that a driver sends directly, for example with `Instrument.write` or `Instrument.ask` in a driver method, are not reported. Use the instrument's logger (`visa_log` for VISA instruments) to see the communication on the wire.

In [14]:
dac.close()
dmm.close()